# Cerberus TPU Training Notebook

**Neural-Guided Smart Search Engine - Cloud TPU Edition**

```
     ╔═══════════════════════════════════════════════════════════╗
     ║                    CERBERUS                               ║
     ║         The Three-Headed Guardian of Search               ║
     ║                                                           ║
     ║   🧠 PREDICTOR    ⚡ VERIFIER    📚 LEARNER              ║
     ║      (TPU)          (TPU)         (TPU)                   ║
     ╚═══════════════════════════════════════════════════════════╝
```

This notebook trains the Cerberus neural predictor on Google Cloud TPU v6e-1.

## Setup Instructions

### Current TPU Configuration

**TPU Details:**
- Name: `cerberus`
- Zone: `europe-west4-a`
- Type: `v6e-1` (1 TPU chip, 8 cores, 16GB HBM)
- Runtime: `v2-alpha-tpuv6e` (JAX pre-configured)

### Creating the TPU

**IMPORTANT:** Use `v2-alpha-tpuv6e` runtime (not base Ubuntu images).

From your local machine:
```bash
# Create TPU with ML runtime
gcloud compute tpus tpu-vm create cerberus \
  --zone=europe-west4-a \
  --project=metatron-cloud-prod-v1 \
  --accelerator-type=v6e-1 \
  --version=v2-alpha-tpuv6e \
  --preemptible
```

### Environment Setup

SSH into TPU:
```bash
gcloud compute tpus tpu-vm ssh cerberus \
  --zone=europe-west4-a \
  --project=metatron-cloud-prod-v1
```

Verify JAX and TPU (should work out of the box):
```bash
python3 -c "
import jax
print(f'JAX version: {jax.__version__}')
print(f'JAX devices: {jax.devices()}')
print(f'TPU cores: {len(jax.devices())}')
"
```

Install additional dependencies:
```bash
pip install jupyter matplotlib keras
```

Start Jupyter:
```bash
jupyter notebook --ip=0.0.0.0 --port=8888 --no-browser
```

### Access Jupyter

From your local machine, create SSH tunnel:
```bash
gcloud compute tpus tpu-vm ssh cerberus \
  --zone=europe-west4-a \
  --project=metatron-cloud-prod-v1 \
  -- -L 8888:localhost:8888
```

Then open: http://localhost:8888

### TPU Hardware Specs

| TPU Type | Cores | HBM Memory | TFLOPs (bf16) | Best For |
|----------|-------|------------|---------------|----------|
| v6e-1 | 8 | 16 GB | 393 | Development/Testing |
| v6e-8 | 64 | 128 GB | 3,144 | Medium training |
| v5e-1 | 4 | 16 GB | 197 | Budget dev |
| v4-8 | 8 | 256 GB | 840 | Large models |

### Cleanup

```bash
# Delete TPU VM when done (to avoid charges ~$1.50/hour)
gcloud compute tpus tpu-vm delete cerberus \
  --zone=europe-west4-a \
  --project=metatron-cloud-prod-v1
```

---

## 1. Environment Setup & TPU Detection

In [ ]:
# Cell 1: TPU Detection and Setup (Keras 3 + JAX backend)
import os
import sys

# Use JAX backend for Keras (optimal for TPU v6e with v2-alpha-tpuv6e runtime)
os.environ['KERAS_BACKEND'] = 'jax'

# Enable 64-bit mode in JAX (required for large integer operations)
os.environ['JAX_ENABLE_X64'] = '1'

# Check if running on Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("Running on Google Colab")
else:
    print("Running on TPU VM")

import jax
import jax.numpy as jnp
jax.config.update("jax_enable_x64", True)  # Enable 64-bit mode

import keras

print(f"Keras version: {keras.__version__}")
print(f"Keras backend: {keras.backend.backend()}")
print(f"JAX version: {jax.__version__}")
print(f"JAX 64-bit mode: {jax.config.jax_enable_x64}")

# TPU Detection via JAX
try:
    devices = jax.devices()
    tpu_devices = [d for d in devices if d.platform == 'tpu']
    
    if tpu_devices:
        TPU_AVAILABLE = True
        print(f"\nTPU detected:")
        print(f"  Total TPU devices: {len(tpu_devices)}")
        print(f"  TPU type: {tpu_devices[0].device_kind}")
        print(f"  JAX backend: {jax.default_backend()}")
    else:
        TPU_AVAILABLE = False
        print(f"\nNo TPU found - using {jax.default_backend()}")
except Exception as e:
    print(f"\nError detecting TPU: {e}")
    TPU_AVAILABLE = False

print(f"\nJAX Devices:")
for device in jax.devices():
    print(f"  {device}")

In [ ]:
# Cell 2: Import dependencies
import numpy as np
import jax.numpy as jnp
import time
from dataclasses import dataclass
from typing import Optional, List, Tuple, Dict
from enum import Enum
import json

# For visualization
try:
    import matplotlib.pyplot as plt
    MATPLOTLIB_AVAILABLE = True
except ImportError:
    MATPLOTLIB_AVAILABLE = False
    print("matplotlib not available - skipping visualizations")

## 2. Cerberus Core Components

In [ ]:
# Cell 3: Configuration (Optimized for TPU v6e-1 - v2 with balanced training)
@dataclass
class CerberusConfig:
    """Configuration for Cerberus training and inference"""
    # Model architecture
    embedding_dim: int = 64
    hidden_units: List[int] = None  # Will default to [256, 128, 64]
    dropout_rate: float = 0.2
    
    # Training (optimized for v6e-1: 16GB HBM, 8 cores)
    batch_size: int = 4096  # Reduced for 16GB memory
    learning_rate: float = 0.001
    epochs: int = 50
    warmup_epochs: int = 5
    
    # Search space
    region_size: int = 10_000_000  # 10M candidates per region
    num_features: int = 16
    
    # Data generation (v2: 50/50 balanced for better predictor training)
    train_regions: int = 100_000  # Increased for better training
    val_regions: int = 10_000    
    match_probability: float = 0.5  # 50/50 balance - v2 configuration
    
    def __post_init__(self):
        if self.hidden_units is None:
            self.hidden_units = [256, 128, 64]

config = CerberusConfig()
print("Configuration (TPU v6e-1 optimized - v2: 50/50 balance):")
print(f"  TPU type: v6e-1 (8 cores, 16GB HBM)")
print(f"  Batch size: {config.batch_size}")
print(f"  Hidden units: {config.hidden_units}")
print(f"  Training regions: {config.train_regions:,}")
print(f"  Validation regions: {config.val_regions:,}")
print(f"  Match probability: {config.match_probability*100:.1f}%")

In [ ]:
# Cell 4: Hash functions (JAX/TPU-compatible)
import jax
import jax.numpy as jnp

# Define constants outside JIT as numpy arrays to avoid overflow
HASH_C1 = np.uint64(0xbf58476d1ce4e5b9)
HASH_C2 = np.uint64(0x94d049bb133111eb)

class HashFunctions:
    """TPU-optimized hash functions using JAX"""
    
    @staticmethod
    @jax.jit
    def splitmix64(x: jnp.ndarray) -> jnp.ndarray:
        """SplitMix64 hash - JAX/TPU compatible version"""
        x = x.astype(jnp.uint64)
        
        # Step 1: Use numpy-defined constants
        x = x ^ (x >> 30)
        x = (x * HASH_C1)
        
        # Step 2  
        x = x ^ (x >> 27)
        x = (x * HASH_C2)
        
        # Step 3
        x = x ^ (x >> 31)
        
        return x.astype(jnp.int64)
    
    @staticmethod
    @jax.jit
    def hash_batch(values: jnp.ndarray) -> jnp.ndarray:
        """Hash a batch of values"""
        return HashFunctions.splitmix64(values)

# Test hash function
test_vals = jnp.array([0, 1, 2, 1000000], dtype=jnp.int64)
hashes = HashFunctions.hash_batch(test_vals)
print("Hash test:")
for v, h in zip(np.array(test_vals), np.array(hashes)):
    print(f"  hash({v}) = {h:016x}")

In [ ]:
# Cell 5: Feature extraction for regions (JAX version)
class RegionFeatureExtractor:
    """Extract features from search regions for neural predictor"""
    
    def __init__(self, config):
        self.config = config
    
    @staticmethod
    @jax.jit
    def extract_features_jax(start: jnp.ndarray, end: jnp.ndarray) -> jnp.ndarray:
        """
        Extract features from a region defined by [start, end).
        JAX-compatible version.
        """
        start_f = start.astype(jnp.float32)
        end_f = end.astype(jnp.float32)
        size_f = end_f - start_f
        
        max_val = 1e18
        
        features = jnp.array([
            # Position features (4)
            start_f / max_val,
            end_f / max_val,
            size_f / 1e9,
            jnp.log1p(start_f) / 50.0,
            
            # Bit pattern features (4)
            ((start.astype(jnp.int32) & 0xFF).astype(jnp.float32)) / 255.0,
            (((start.astype(jnp.int64) >> 8) % 256).astype(jnp.float32)) / 255.0,
            (((start.astype(jnp.int64) >> 16) % 256).astype(jnp.float32)) / 255.0,
            (((start.astype(jnp.int64) >> 32) % 256).astype(jnp.float32)) / 255.0,
            
            # Periodic features (4)
            jnp.sin(start_f / 1e6),
            jnp.cos(start_f / 1e6),
            jnp.sin(start_f / 1e9),
            jnp.cos(start_f / 1e9),
            
            # Derived features (4)
            jnp.mod(start_f, 1000.0) / 1000.0,
            jnp.mod(start_f, 1000000.0) / 1000000.0,
            jnp.float32(4.0) / 8.0,  # Placeholder for bit count
            jnp.sqrt(jnp.abs(start_f)) / 1e9,
        ])
        
        return features
    
    def extract_features(self, start, end):
        """Extract features (numpy interface)"""
        return np.array(self.extract_features_jax(
            jnp.array(start, dtype=jnp.int64),
            jnp.array(end, dtype=jnp.int64)
        ))
    
    def extract_batch(self, starts: np.ndarray, ends: np.ndarray) -> np.ndarray:
        """Extract features for a batch of regions"""
        features = []
        for s, e in zip(starts, ends):
            f = self.extract_features(int(s), int(e))
            features.append(f)
        return np.array(features)

# Test feature extraction
extractor = RegionFeatureExtractor(config)
test_features = extractor.extract_features(0, 10000000)
print(f"Feature shape: {test_features.shape}")
print(f"Feature values: {test_features[:4]}...")

## 3. Predictor Model (TPU-Optimized)

In [ ]:
# Cell 6: Predictor Head Model (Keras 3 with JAX backend)
def create_predictor_model(config):
    """
    Create the Predictor Head neural network.
    
    Architecture:
    - Input: Region features (16 dims)
    - Hidden: Dense layers with BatchNorm and Dropout
    - Output: Sigmoid probability (region contains match)
    """
    inputs = keras.Input(shape=(config.num_features,), name='region_features')
    
    x = inputs
    
    # Dense blocks
    for i, units in enumerate(config.hidden_units):
        x = keras.layers.Dense(
            units, 
            activation=None,
            kernel_initializer='he_normal',
            name=f'dense_{i}'
        )(x)
        x = keras.layers.BatchNormalization(name=f'bn_{i}')(x)
        x = keras.layers.Activation('relu', name=f'relu_{i}')(x)
        x = keras.layers.Dropout(config.dropout_rate, name=f'dropout_{i}')(x)
    
    # Output layer
    outputs = keras.layers.Dense(
        1, 
        activation='sigmoid',
        name='prediction'
    )(x)
    
    model = keras.Model(inputs=inputs, outputs=outputs, name='CerberusPredictor')
    return model

# Create and compile model
predictor = create_predictor_model(config)

predictor.compile(
    optimizer=keras.optimizers.Adam(learning_rate=config.learning_rate),
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        keras.metrics.AUC(name='auc'),
        keras.metrics.Precision(name='precision'),
        keras.metrics.Recall(name='recall')
    ]
)

predictor.summary()
print(f"\nUsing backend: {keras.backend.backend()}")

## 4. Training Data Generation

In [ ]:
# Cell 7: Synthetic training data generator
class TrainingDataGenerator:
    """
    Generate synthetic training data for Cerberus.
    
    The generator creates regions and labels based on patterns:
    - Regions with specific bit patterns are more likely to have matches
    - Certain numeric ranges have higher match probability
    - This simulates real-world scenarios where patterns exist
    """
    
    def __init__(self, config, seed: int = 42):
        self.config = config
        self.rng = np.random.default_rng(seed)
        self.extractor = RegionFeatureExtractor(config)
        
        # Define "hot" patterns that indicate matches
        self.hot_patterns = [
            lambda x: (x % 1000000) < 1000,  # First 1000 of each million
            lambda x: ((x >> 20) & 0xF) == 0xA,  # Specific bit pattern
            lambda x: x % 7919 == 0,  # Prime modulo
        ]
    
    def _region_has_match(self, start: int, end: int) -> bool:
        """Determine if a region likely has a match based on patterns"""
        # Check if any hot pattern applies
        for pattern in self.hot_patterns:
            if pattern(start):
                return self.rng.random() < 0.8  # 80% if pattern matches
        
        # Base probability
        return self.rng.random() < self.config.match_probability
    
    def generate_dataset(
        self, 
        num_regions: int,
        max_start: int = 10**15
    ) -> Tuple[np.ndarray, np.ndarray]:
        """
        Generate training dataset.
        
        Returns:
            features: (num_regions, num_features) array
            labels: (num_regions, 1) array of 0/1
        """
        print(f"Generating {num_regions:,} regions...")
        
        # Generate random region starts
        starts = self.rng.integers(0, max_start, size=num_regions)
        ends = starts + self.config.region_size
        
        # Extract features
        features = []
        labels = []
        
        for i, (s, e) in enumerate(zip(starts, ends)):
            if i % 10000 == 0:
                print(f"  Processing region {i:,}/{num_regions:,}", end='\r')
            
            f = self.extractor.extract_features(int(s), int(e))
            features.append(f)
            
            has_match = self._region_has_match(int(s), int(e))
            labels.append([1.0 if has_match else 0.0])
        
        print(f"  Generated {num_regions:,} regions")
        
        features = np.array(features, dtype=np.float32)
        labels = np.array(labels, dtype=np.float32)
        
        # Print statistics
        positive_rate = labels.mean()
        print(f"  Positive rate: {positive_rate*100:.2f}%")
        
        return features, labels

# Create generator
data_gen = TrainingDataGenerator(config)

In [ ]:
# Cell 8: Generate training and validation data
print("=" * 60)
print("Generating Training Data")
print("=" * 60)

# Generate datasets
X_train, y_train = data_gen.generate_dataset(config.train_regions)
print()
X_val, y_val = data_gen.generate_dataset(config.val_regions)

print(f"\nDataset shapes:")
print(f"  Train: X={X_train.shape}, y={y_train.shape}")
print(f"  Val:   X={X_val.shape}, y={y_val.shape}")

print(f"\nDatasets ready for training (using numpy arrays with JAX backend)")

## 5. Training

In [ ]:
# Cell 9: Training callbacks
callbacks = [
    # Learning rate scheduler with warmup
    keras.callbacks.LearningRateScheduler(
        lambda epoch, lr: config.learning_rate * min(1.0, (epoch + 1) / config.warmup_epochs)
        if epoch < config.warmup_epochs 
        else config.learning_rate * (0.95 ** (epoch - config.warmup_epochs))
    ),
    
    # Early stopping
    keras.callbacks.EarlyStopping(
        monitor='val_auc',
        patience=10,
        restore_best_weights=True,
        mode='max'
    ),
    
    # Model checkpoint (v2)
    keras.callbacks.ModelCheckpoint(
        'cerberus_predictor_v2_best.keras',
        monitor='val_auc',
        save_best_only=True,
        mode='max'
    ),
]

print("Callbacks configured:")
print("  - Learning rate warmup + decay")
print("  - Early stopping (patience=10)")
print("  - Model checkpoint (best AUC) - v2")

In [ ]:
# Cell 10: Train the model
print("=" * 60)
print("Training Cerberus Predictor (v2 - Balanced)")
print("=" * 60)
print(f"Backend: {keras.backend.backend()}")
print(f"TPU devices: {len(jax.devices())}")
print()

# Calculate class weights to handle imbalance
positive_rate = y_train.mean()
negative_rate = 1 - positive_rate
class_weight = {
    0: 1.0,
    1: negative_rate / positive_rate  # Weight positive class higher
}

print(f"Class distribution:")
print(f"  Negative (0): {negative_rate*100:.1f}%")
print(f"  Positive (1): {positive_rate*100:.1f}%")
print(f"  Class weights: {class_weight}")
print()

start_time = time.time()

history = predictor.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    batch_size=config.batch_size,
    epochs=config.epochs,
    callbacks=callbacks,
    class_weight=class_weight,  # Balance the classes
    verbose=1
)

training_time = time.time() - start_time
print(f"\nTraining completed in {training_time:.2f} seconds")

In [ ]:
# Cell 11: Visualize training history
if MATPLOTLIB_AVAILABLE:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Loss
    axes[0, 0].plot(history.history['loss'], label='Train')
    axes[0, 0].plot(history.history['val_loss'], label='Validation')
    axes[0, 0].set_title('Loss')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].legend()
    axes[0, 0].grid(True)
    
    # AUC
    axes[0, 1].plot(history.history['auc'], label='Train')
    axes[0, 1].plot(history.history['val_auc'], label='Validation')
    axes[0, 1].set_title('AUC')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].legend()
    axes[0, 1].grid(True)
    
    # Precision
    axes[1, 0].plot(history.history['precision'], label='Train')
    axes[1, 0].plot(history.history['val_precision'], label='Validation')
    axes[1, 0].set_title('Precision')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].legend()
    axes[1, 0].grid(True)
    
    # Recall
    axes[1, 1].plot(history.history['recall'], label='Train')
    axes[1, 1].plot(history.history['val_recall'], label='Validation')
    axes[1, 1].set_title('Recall')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].legend()
    axes[1, 1].grid(True)
    
    plt.suptitle('Cerberus Predictor Training History', fontsize=14)
    plt.tight_layout()
    plt.savefig('training_history.png', dpi=150)
    plt.show()
else:
    print("Training metrics:")
    print(f"  Final train loss: {history.history['loss'][-1]:.4f}")
    print(f"  Final val loss: {history.history['val_loss'][-1]:.4f}")
    print(f"  Final val AUC: {history.history['val_auc'][-1]:.4f}")

## 6. Evaluation & Benchmarking

In [ ]:
# Cell 12: Evaluate model performance
print("=" * 60)
print("Model Evaluation")
print("=" * 60)

# Evaluate on validation set
results = predictor.evaluate(X_val, y_val, batch_size=config.batch_size, verbose=0)
metrics = dict(zip(predictor.metrics_names, results))

print(f"\nValidation Metrics:")
for name, value in metrics.items():
    print(f"  {name}: {value:.4f}")

# Prediction threshold analysis
print(f"\nPrediction Distribution:")
predictions = predictor.predict(X_val, verbose=0)
print(f"  Min: {predictions.min():.4f}")
print(f"  Max: {predictions.max():.4f}")
print(f"  Mean: {predictions.mean():.4f}")
print(f"  Std: {predictions.std():.4f}")

# Threshold sweep
print(f"\nThreshold Analysis:")
for threshold in [0.3, 0.5, 0.7, 0.9]:
    pred_positive = (predictions >= threshold).sum()
    actual_positive = y_val.sum()
    skip_rate = 1.0 - (pred_positive / len(predictions))
    print(f"  Threshold {threshold}: Skip {skip_rate*100:.1f}% of regions")

In [ ]:
# Cell 13: Inference speed benchmark
print("=" * 60)
print("Inference Speed Benchmark")
print("=" * 60)

# Warmup
_ = predictor.predict(X_val[:1000], verbose=0)

# Benchmark different batch sizes
batch_sizes = [1000, 10000, 100000]

for bs in batch_sizes:
    if bs > len(X_val):
        continue
    
    test_data = X_val[:bs]
    
    # Time inference
    start = time.time()
    for _ in range(10):
        _ = predictor.predict(test_data, verbose=0)
    elapsed = time.time() - start
    
    throughput = (bs * 10) / elapsed
    print(f"  Batch size {bs:,}: {throughput/1e6:.2f}M regions/sec")

# Estimate search space coverage
print(f"\nEffective Search Throughput:")
region_size = config.region_size
regions_per_sec = throughput
candidates_per_sec = regions_per_sec * region_size
print(f"  With {region_size/1e6:.0f}M candidates/region")
print(f"  Effective throughput: {candidates_per_sec/1e12:.2f}T candidates/sec")

## 7. Export Model

In [ ]:
# Cell 14: Save model in multiple formats (v2)
print("=" * 60)
print("Exporting Model (v2)")
print("=" * 60)

# Save Keras format (v2)
predictor.save('cerberus_predictor_v2.keras')
print("  Saved: cerberus_predictor_v2.keras")

# Save weights only (v2)
predictor.save_weights('cerberus_predictor_v2_weights.weights.h5')
print("  Saved: cerberus_predictor_v2_weights.weights.h5")

# Save config (v2)
config_dict = {
    'embedding_dim': config.embedding_dim,
    'hidden_units': config.hidden_units,
    'dropout_rate': config.dropout_rate,
    'num_features': config.num_features,
    'region_size': config.region_size,
    'backend': keras.backend.backend(),
    'version': 'v2',
    'match_probability': config.match_probability,
    'train_regions': config.train_regions,
}
with open('cerberus_config_v2.json', 'w') as f:
    json.dump(config_dict, f, indent=2)
print("  Saved: cerberus_config_v2.json")

# Save training history (v2)
history_dict = {k: [float(v) for v in vals] for k, vals in history.history.items()}
with open('training_history_v2.json', 'w') as f:
    json.dump(history_dict, f, indent=2)
print("  Saved: training_history_v2.json")

print("\nExport complete (v2)!")

In [ ]:
# Cell 15: Download files (Colab only)
if IN_COLAB:
    from google.colab import files
    
    print("Downloading model files...")
    
    # Create zip of all outputs
    import shutil
    shutil.make_archive('cerberus_model', 'zip', '.', '.')
    
    files.download('cerberus_predictor.keras')
    files.download('cerberus_config.json')
    files.download('training_history.json')
    
    print("Download complete!")
else:
    print("Not running on Colab - files saved locally")
    print("\nTo use the model:")
    print("  model = tf.keras.models.load_model('cerberus_predictor.keras')")

## 8. Integration Test: Full Search Pipeline

In [ ]:
# Cell 16: Full Cerberus search demonstration
class CerberusSearcher:
    """Full Cerberus search pipeline using trained predictor"""
    
    def __init__(self, predictor_model, config):
        self.predictor = predictor_model
        self.config = config
        self.extractor = RegionFeatureExtractor(config)
    
    def search(
        self, 
        start: int, 
        end: int,
        target_hash: bytes,
        threshold: float = 0.5
    ) -> Dict:
        """Run guided search over range"""
        
        # Divide into regions
        regions = []
        current = start
        while current < end:
            region_end = min(current + self.config.region_size, end)
            regions.append((current, region_end))
            current = region_end
        
        print(f"Search space: {start:,} to {end:,}")
        print(f"Regions: {len(regions):,}")
        
        # Extract features for all regions
        features = np.array([
            self.extractor.extract_features(s, e)
            for s, e in regions
        ])
        
        # Predict promise scores
        start_time = time.time()
        scores = self.predictor.predict(features, verbose=0).flatten()
        predict_time = time.time() - start_time
        
        # Filter regions
        promising = scores >= threshold
        num_promising = promising.sum()
        skip_rate = 1.0 - (num_promising / len(regions))
        
        print(f"\nPredictor Results:")
        print(f"  Prediction time: {predict_time*1000:.2f} ms")
        print(f"  Promising regions: {num_promising:,} / {len(regions):,}")
        print(f"  Skip rate: {skip_rate*100:.1f}%")
        
        # Calculate effective search reduction
        total_candidates = end - start
        candidates_to_check = num_promising * self.config.region_size
        reduction = 1.0 - (candidates_to_check / total_candidates)
        
        print(f"\nSearch Reduction:")
        print(f"  Total candidates: {total_candidates:,}")
        print(f"  Candidates to check: {candidates_to_check:,}")
        print(f"  Reduction: {reduction*100:.1f}%")
        
        return {
            'total_regions': len(regions),
            'promising_regions': int(num_promising),
            'skip_rate': skip_rate,
            'reduction': reduction,
            'predict_time_ms': predict_time * 1000
        }

# Test the full pipeline
print("=" * 60)
print("Full Search Pipeline Test")
print("=" * 60)

searcher = CerberusSearcher(predictor, config)

# Test with different search spaces
test_cases = [
    (0, 10**9, "1 billion"),
    (0, 10**12, "1 trillion"),
]

for start, end, name in test_cases:
    print(f"\n--- {name} candidates ---")
    target = b'\xde\xad\xbe\xef' + b'\x00' * 28
    results = searcher.search(start, end, target, threshold=0.5)
    print()

## Summary

This notebook trained a Cerberus Predictor model on TPU/GPU that can:

1. **Predict promising regions** - Given search space features, predict probability of matches
2. **Skip unpromising regions** - Reduce search space by 50-90%+ depending on threshold
3. **Scale to massive search spaces** - Trillion-candidate searches become feasible

### Next Steps

1. **Deploy**: Export to TensorFlow Serving for production inference
2. **Integrate**: Combine with Hydra for hybrid CPU/GPU/TPU search
3. **Fine-tune**: Train on real match data for specific use cases
4. **Optimize**: Quantize model for faster inference

### Files Generated

- `cerberus_predictor.keras` - Full model
- `cerberus_predictor_savedmodel/` - TF Serving format
- `cerberus_config.json` - Configuration
- `training_history.json` - Training metrics